# Lab 2.3 - Quantization: Post-Training vs Quantization-Aware Training

This notebook has `# TODO` markers (TODO 1-6). Work through them in order.

Copy your `mnist_cnn.pth` from Lab 2.2 into this same folder before you start.

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import datasets
import litert_torch
import ai_edge_litert.interpreter as tflite
import matplotlib.pyplot as plt

torch.manual_seed(10)
np.random.seed(10)

print("PyTorch version:", torch.__version__)

## Step 0 - Load data and the Lab 2.2 model

Same MNIST preprocessing as Lab 2.2 (normalize to [0,1], add channel dim in NCHW format).
This part is provided for you.

Note: loaded with `torch.load("mnist_cnn.pth")` matching the model built in Lab 2.2.

In [ ]:
train_dataset = datasets.MNIST(root="./data", train=True, download=True)
test_dataset  = datasets.MNIST(root="./data", train=False, download=True)
x_train = train_dataset.data.numpy()
y_train = train_dataset.targets.numpy()
x_test  = test_dataset.data.numpy()
y_test  = test_dataset.targets.numpy()

x_train_n = (x_train / 255.0).astype("float32")[:, np.newaxis, :, :]
x_test_n  = (x_test  / 255.0).astype("float32")[:, np.newaxis, :, :]

def build_model():
    return nn.Sequential(
        nn.Conv2d(1, 8, kernel_size=3),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(8, 16, kernel_size=3),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Flatten(),
        nn.Linear(16 * 5 * 5, 32),
        nn.ReLU(),
        nn.Linear(32, 10),
    )

model = build_model()
model.load_state_dict(torch.load("mnist_cnn.pth", weights_only=True))
model.eval()
print(model)

## Step 1 - Baseline

**TODO 1:** Record the float32 model's size on disk (`os.path.getsize`, in KB)
and its test accuracy. Also measure latency: the average
time for a single-image forward pass, calling the model directly
(`model(torch.from_numpy(x))` with `torch.no_grad()`), over 100 images.

In [ ]:
# TODO 1: size (KB), accuracy, and per-image latency (ms) of the baseline model
baseline_size_kb = None
baseline_acc = None

n_latency = 100
start = time.time()
for i in range(n_latency):
    pass  # TODO: call the model on a single image, torch.from_numpy(x_test_n[i:i+1])
baseline_latency_ms = None

print(f"Baseline float32 -- size: {baseline_size_kb:.1f} KB, "
      f"accuracy: {baseline_acc:.4f}, latency: {baseline_latency_ms:.3f} ms/image")

## Step 2 - Post-Training Quantization (PTQ)

Use `litert_torch.convert` to convert the PyTorch model to LiteRT (`.tflite`) format:

```python
sample_inputs = (torch.randn(1, 1, 28, 28),)
edge_model = litert_torch.convert(model.eval(), sample_inputs)
edge_model.export("ptq_model.tflite")
```

**TODO 2:** Convert the model to LiteRT format and save as `ptq_model.tflite`.

In [ ]:
# TODO 2: convert model with litert_torch, export to "ptq_model.tflite", and load bytes
tflite_ptq = None

# edge_model.export("ptq_model.tflite") saves directly to disk.
# Read the exported model bytes for evaluation:
with open("ptq_model.tflite", "rb") as f:
    tflite_ptq = f.read()

ptq_size_kb = len(tflite_ptq) / 1024
print(f"PTQ model size: {ptq_size_kb:.1f} KB "
      f"({baseline_size_kb / ptq_size_kb:.1f}x smaller than float32)")

**Evaluating a TFLite model is different from evaluating a PyTorch model** -
you drive the `Interpreter` (`ai_edge_litert.interpreter.Interpreter`) directly.
If the input tensor is `int8`, you quantize each image using the input tensor's
`(scale, zero_point)` before feeding it in:

```
quantized_pixel = round(pixel_value / scale + zero_point)
```

If the tensor is float32 (`scale == 0`), pass the float pixels directly.
The output is evaluated using `argmax` on the predicted scores.

**TODO 3:** Complete `evaluate_tflite()`: quantize each input (if quantized),
run it through the interpreter, and check whether the predicted class matches
the true label. Also time 100 invocations for a latency figure, the same way as
Step 1.

In [ ]:
def evaluate_tflite(tflite_bytes, x, y, n_latency=100):
    interpreter = tflite.Interpreter(model_content=tflite_bytes)
    interpreter.allocate_tensors()
    inp = interpreter.get_input_details()[0]
    out = interpreter.get_output_details()[0]
    scale, zero_point = inp["quantization"]

    def quantize(img):
        # TODO 3a: quantize a float image using (scale, zero_point)
        return None

    # TODO 3b: loop over all of x/y, run inference, count correct predictions
    correct = 0
    accuracy = None

    # TODO 3c: time n_latency invocations (same pattern as Step 1)
    latency_ms = None

    return accuracy, latency_ms, inp["dtype"], out["dtype"]

ptq_acc, ptq_latency_ms, ptq_in_dtype, ptq_out_dtype = evaluate_tflite(tflite_ptq, x_test_n, y_test)
print(f"PTQ -- accuracy: {ptq_acc:.4f}, latency: {ptq_latency_ms:.3f} ms/image, "
      f"input dtype: {ptq_in_dtype}, output dtype: {ptq_out_dtype}")

## Step 3 - Quantization-Aware Training (QAT)

In PyTorch, quantization-aware training simulates quantization effects during
training, allowing weights to adapt and preserve accuracy.

**TODO 4:** Prepare the model for QAT or fine-tuning, and train for 2 epochs.

In [ ]:
# TODO 4: prepare the model for QAT/fine-tuning and fine-tune for 2 epochs
qat_model = None

**TODO 5:** Convert `qat_model` to a fully-quantized int8 LiteRT model (`qat_model.tflite`)
using `litert_torch.convert()`, and evaluate it with your
`evaluate_tflite()` from Step 2.

In [ ]:
# TODO 5: convert qat_model with litert_torch, export to "qat_model.tflite", and load bytes
tflite_qat = None

with open("qat_model.tflite", "rb") as f:
    tflite_qat = f.read()

qat_size_kb = len(tflite_qat) / 1024
qat_acc, qat_latency_ms, qat_in_dtype, qat_out_dtype = evaluate_tflite(tflite_qat, x_test_n, y_test)
print(f"QAT -- size: {qat_size_kb:.1f} KB, accuracy: {qat_acc:.4f}, "
      f"latency: {qat_latency_ms:.3f} ms/image, "
      f"input dtype: {qat_in_dtype}, output dtype: {qat_out_dtype}")

## Step 4 - Compare

Check that both quantized models have `int8` input/output tensors printed
above - that's the requirement Lab 2.4's microcontroller runtime needs.

**TODO 6:** Build a comparison table (a small `pandas.DataFrame` is easiest)
with one row per variant (float32 baseline, PTQ int8, QAT int8) and columns
for size (KB), accuracy, and latency (ms). Then plot size and accuracy side
by side as bar charts.

In [ ]:
# TODO 6: build the comparison DataFrame
comparison = None
comparison

In [ ]:
# TODO 6 (continued): bar charts for size and accuracy, side by side
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

plt.tight_layout()
plt.savefig("quantization_comparison.png", dpi=110)
plt.show()

## Discussion

In one paragraph: did QAT actually beat PTQ here? By how much, and was the
extra training time worth it for your use case?